In [ ]:
# Importing neccesary libraries
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
import os
import warnings
warnings.filterwarnings("ignore")

os.listdir('../data')


In [ ]:
# Reading the dataset
dataset = pd.read_csv("../data/smartphones.csv")
data_copia = dataset.copy()
dataset.sample(5)

In [ ]:
dataset.info()

In [ ]:
dataset.describe()

In [ ]:
print(dataset["Storage"].value_counts())
print("-------------------------------")
print(dataset["RAM"].value_counts())

In [ ]:
# Storage distribution
s_Storage = sns.countplot(x="Storage", data=dataset, palette="viridis")


In [ ]:
# RAM distribution
s_RAM = sns.countplot(x="RAM", data=dataset, palette="magma")

In [ ]:
# Defining a function to check missing values. It will return the total and percentage of missing values in each column.
def missing_values(dataset):
    total = dataset.isnull().sum().sort_values(ascending=False)
    percent = (dataset.isnull().sum()/dataset.isnull().count()).sort_values(ascending=False)
    return pd.concat([total, percent], axis=1, keys=["Total", "Percent"])

missing_values(dataset)

In [ ]:
# Imputing missing values using SimpleImputer
imputer = SimpleImputer(strategy="most_frequent")
dataset[["Storage", "RAM"]] = imputer.fit_transform(dataset[["Storage", "RAM"]])
dataset.info()

In [ ]:
# Visualizing the outliers using boxplots
plt.figure(figsize=(10, 5))
plt.boxplot(dataset[["Storage", "RAM", "Final Price"]], vert=False, patch_artist=True, labels=["Storage", "RAM", "Final Price"], boxprops=dict(facecolor="coral", color="black"), medianprops=dict(color="black"), whiskerprops=dict(color="black"), capprops=dict(color="black"))
plt.title("Boxplot of Storage, RAM and Final Price")
plt.xlabel("Features")
plt.ylabel("Values")
plt.show()


In [ ]:
# Removing outliers using IQR method
Q1 = dataset[["Storage", "Final Price"]].quantile(0.25)
Q3 = dataset[["Storage", "Final Price"]].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
dataset = dataset[~((dataset[["Storage", "Final Price"]] < lower_bound) | (dataset[["Storage", "Final Price"]] > upper_bound)).any(axis=1)]   
dataset.info()

In [ ]:
# Understanding if the data is skewed or not using histograms
plt.figure(figsize=(15, 5))
plt.subplot(1, 3, 1)
sns.histplot(dataset["Storage"], kde=True, color="skyblue")
plt.subplot(1, 3, 2)
sns.histplot(dataset["RAM"], kde=True, color="salmon")
plt.subplot(1, 3, 3)
sns.histplot(dataset["Final Price"], kde=True, color="lightgreen")
plt.tight_layout()
plt.show()

In [ ]:
dataset[["Storage", "RAM", "Final Price"]].skew()

In [ ]:
# Managing the skewness of the data using log transformation
dataset["Final Price"] = np.log1p(dataset["Final Price"])
dataset["Storage"] = np.log1p(dataset["Storage"])
dataset[["Storage", "RAM", "Final Price"]].skew()


In [ ]:
# Change the column free into boolean values
dataset["Free"] = dataset["Free"].map({"Yes": 1, "No": 0})

In [ ]:
# Dealing with Model using frequency encoding
dataset["Model"] = dataset["Model"].map(dataset["Model"].value_counts())

In [ ]:
# Use frequency encoding for the smartphone column and create new binary features for common keywords in the smartphone names. 
# Then drop the original smartphone column.
dataset["Smartphone_Freq"] = dataset["Smartphone"].map(dataset["Smartphone"].value_counts())
dataset["is_pro"] = dataset["Smartphone"].str.contains("Pro", case=False).astype(int)
dataset["is_plus"] = dataset["Smartphone"].str.contains("Plus", case=False).astype(int)
dataset["is_max"] = dataset["Smartphone"].str.contains("Max", case=False).astype(int)
dataset["is_mini"] = dataset["Smartphone"].str.contains("Mini", case=False).astype(int)
dataset["is_ultra"] = dataset["Smartphone"].str.contains("Ultra", case=False).astype(int)
dataset["is_lite"] = dataset["Smartphone"].str.contains("Lite", case=False).astype(int)
dataset["name_length"] = dataset["Smartphone"].str.split().str.len()
dataset.drop("Smartphone", axis=1, inplace=True)

In [ ]:
# Manage brand and Color columns using one-hot encoding
dataset = pd.get_dummies(dataset, columns=["Brand", "Color"], drop_first=True)
dataset.info()

In [ ]:
dataset.to_csv("../data/smartphones_cleaned.csv", index=False) # Save the cleaned dataset to a new CSV file
